In [ ]:
# Run from the repository root, or open this notebook anywhere inside it.
from pathlib import Path
import os
_repo_candidates = [Path.cwd(), *Path.cwd().parents]
_project_root = next((p for p in _repo_candidates if (p / "code/data_process").is_dir() and (p / "README.md").is_file()), None)
if _project_root is None:
    raise RuntimeError("Open this notebook from inside the repository.")
os.chdir(_project_root)


In [ ]:
import pandas as pd

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
import pandas as pd

# File paths
lihtc_fp = "./legacy_inputs/LIHTC/data_derivatives/lihtc_treatment_panel_newbuild_fullcum_2010_2019.csv"
controls_fp = "./legacy_inputs/LIHTC/data_raw/2010_census_data/nhgis0004_csv/tract_controls_2010_from_nhgis.csv"
lodes_fp = "./legacy_inputs/LODES/data_derivatives/lodes7_tract_agg_2010_2019/lodes_panel_rac_wac_od_2010_2019.parquet"

# Load datasets
lihtc = pd.read_csv(lihtc_fp, dtype={"tract_id": str})
controls = pd.read_csv(controls_fp)
lodes = pd.read_parquet(lodes_fp)

In [ ]:
print("LIHTC dtypes: \n",lihtc.dtypes)
print("controls dtypes: \n",controls.dtypes)
print("lodes dtypes: \n",lodes.dtypes)


In [ ]:
# -------------------------
# Standardize keys
# -------------------------
# IDs to 11-digit strings
def z11(x): return x.astype(str).str.zfill(11)

lodes["tract_id"] = z11(lodes["tract_id"])
lihtc["tract_id"] = z11(lihtc["tract_id"])
controls["geoid"] = z11(controls["geoid"])

# Year to int
lodes["year"] = lodes["year"].astype(int)
lihtc["year"] = lihtc["year"].astype(int)

# -------------------------
# Build panel (base = LODES so untreated tracts remain)
# -------------------------
panel = lodes.merge(
    lihtc,
    on=["tract_id", "year"],
    how="left",
    validate="m:1"  # many LODES rows to at most 1 LIHTC row per tract-year
).merge(
    controls,
    left_on="tract_id", right_on="geoid",
    how="left",
    validate="m:1"
).drop(columns=["geoid"])  # keep only tract_id

# -------------------------
# Fill sensible defaults for untreated years
# -------------------------
# These should be zeros when no LIHTC activity that year / ever
zero_cols = [
    "ever_treated", "treated",
    "n_new_projects", "n_new_units_total", "n_new_units_li",
    "br0_new_units", "br1_new_units", "br2_new_units", "br3_new_units", "br4_new_units",
    "n_units_total_cum_full", "n_units_li_cum_full"
]
for c in zero_cols:
    if c in panel.columns:
        panel[c] = panel[c].fillna(0).astype(int)

# Keep timing as NA for never-treated
for c in ["first_pis_year", "event_time"]:
    if c in panel.columns:
        # leave as NA if missing
        panel[c] = panel[c].astype("Int64")

# -------------------------
# Optional: restrict to 2010–2019 if needed (LODES should already be)
# -------------------------
panel = panel[(panel["year"] >= 2010) & (panel["year"] <= 2019)].copy()

# -------------------------
# Order columns for analysis ergonomics
# -------------------------
id_cols = ["tract_id", "year"]

treat_core = [c for c in ["ever_treated","treated","event_time","first_pis_year"] if c in panel.columns]
treat_intensity = [c for c in [
    "n_new_projects","n_new_units_total","n_new_units_li",
    "br0_new_units","br1_new_units","br2_new_units","br3_new_units","br4_new_units",
    "n_units_total_cum_full","n_units_li_cum_full"
] if c in panel.columns]

# All LODES outcome cols (keep existing order as much as possible)
lodes_outcome_prefixes = ("rac_", "wac_", "od_")
lodes_cols = [c for c in panel.columns if c.startswith(lodes_outcome_prefixes)]

# Controls (2010 cross-section, repeated by year)
control_cols = [
    "pop2010_total","median_age_2010","housing_units_2010","units_total_2010","units_vacant_2010",
    "race_total_2010","white_alone_2010","vacancy_rate_2010","minority_share_2010",
    "median_hh_income_2010","poverty_status_total_2010","poverty_rate_2010",
    "share_high_school_or_higher_2010","share_bachelor_or_higher_2010",
    "STATEA","COUNTYA","TRACTA","GISJOIN"
]
control_cols = [c for c in control_cols if c in panel.columns]

# Everything else (to append at the end)
keep_order = id_cols + treat_core + treat_intensity + lodes_cols + control_cols
other_cols = [c for c in panel.columns if c not in keep_order]
panel = panel[keep_order + other_cols]

# -------------------------
# Basic diagnostics (printed)
# -------------------------
n_rows = len(panel)
n_tracts = panel["tract_id"].nunique()
n_treated = panel.groupby("tract_id")["ever_treated"].max().sum() if "ever_treated" in panel.columns else pd.NA

print(f"Panel rows: {n_rows:,}")
print(f"Unique tracts: {n_tracts:,}")
print(f"Treated tracts (ever): {int(n_treated) if pd.notna(n_treated) else 'NA'}")

# Check duplicates
dups = panel.duplicated(subset=["tract_id","year"]).sum()
print("Duplicate tract_id-year rows:", dups)

# # -------------------------
# # Save
# # -------------------------
# panel.to_parquet(2010_2019_tract_panel_parquet, index=False)
# panel.to_csv(2010_2019_tract_panel_csv, index=False)

# print("Saved:")
# print(" -", 2010_2019_tract_panel_parquet)
# print(" -", 2010_2019_tract_panel_csv)

# -------------------------
# (Optional) Handy weights & scales for later regressions
# Uncomment if useful:
# if "pop2010_total" in panel.columns:
#     panel["weight_pop2010"] = panel["pop2010_total"].clip(lower=1)
#     # Example intensity per 1k residents:
#     if "n_new_units_total" in panel.columns:
#         panel["n_new_units_per_1k"] = 1000 * panel["n_new_units_total"] / panel["weight_pop2010"]


### Rearrange columns

In [ ]:
panel = panel.rename(columns = str.lower)

# Move [STATEA, COUNTYA, TRACTA] (and optionally state) right after [tract_id, year]
move_after = ['state', "statea", "countya", "tracta"]

    

# Reorder columns: keep everything else in original order
cols = list(panel.columns)

# Find indices
id_pos = cols.index("year") + 1  # insert after 'year'
for c in reversed(move_after):   # reverse to maintain order when inserting
    if c in cols:
        cols.insert(id_pos, cols.pop(cols.index(c)))

# Reorder panel
panel = panel[cols]


panel = panel.rename(columns={'pop2010_total': 'pop_total_2010', 'white_alone_2010': 'white_alone_pop_2010'})
panel = panel.rename(columns={'n_units_total_cum_full': 'n_units_total_cum_1987_onwards', 'n_units_li_cum_full': 'n_units_li_cum_1987_onwards'})
panel = panel.rename(columns={'od_same_total': 'od_same_tract_commute_total'})


# Columns to rename
acs_cols = [
    'pop_total_2010','median_age_2010','housing_units_2010','units_total_2010',
    'units_vacant_2010','white_alone_pop_2010','vacancy_rate_2010',
    'minority_share_2010','median_hh_income_2010','poverty_status_total_2010',
    'poverty_rate_2010','share_bachelor_or_higher_2010',
    'share_some_college_or_higher_2010'
]

# Keep only those that exist in panel
acs_cols = [c for c in acs_cols if c in panel.columns]

# Build rename dictionary: remove "_2010", then prefix "acs2010_"
rename_map = {c: "acs2010_" + c.replace("_2010", "") for c in acs_cols}


# Apply renaming
panel = panel.rename(columns=rename_map)

to_drop = ["gisjoin", "race_total_2010", "acs2010_units_total"]
# Drop GISJOIN if present
for col in to_drop:
    if col in panel.columns:
        panel = panel.drop(columns= col)


# Optional check
print(panel.columns)  # preview columns



In [ ]:
panel

In [ ]:
panel.to_csv("./legacy_inputs/LIHTC/data_panels/2010_2019_did_panel.csv")

In [ ]:
print(f"✅next task: reorg the col names: move [state, STATEA, COUNTYA, TRACTA] right after [tract_id, year]."
      "Drop column [GISJOIN] \n")

print("✅next task: validate the merged panel, by comparing selected tract_id, e.g. '01001020300' of 'panel'"
        "with 'lihtc' and 'controls'.  \n")

print("✅go back to lihtc property, check cum totals \n")

print("next task: should the [event_time] be NaNs for those unmatched tracts,"
        "in terms of the DID/event study regression purpose ")


In [ ]:
check_panel_case = panel[panel['tract_id'] == '01003010500']
check_acs_case = controls[controls['geoid'] == '01003010500']
check_lihtc_case = lihtc[lihtc['tract_id'] == '01003010500']
check_lodes_case = lodes[lodes['tract_id'] == '01003010500']


In [ ]:
check_panel_case

In [ ]:
check_lodes_case

In [ ]:
check_lihtc_case

In [ ]:
check_acs_case

In [ ]:
print(panel['ever_treated'].value_counts())

print("There are 582640 obs of which ever_treated = 0. They correspond to the tracts not in LIHTC database but in LODES.")

In [ ]:
print((panel['year'] == 2010).sum())
print((panel['year'] == 2011).sum())
print("=> unbalanced panel: certain tract_id s are not available for some years, in LODES")

In [ ]:
nunique_tract = panel['tract_id'].nunique()
print('unique tracts in merged panel:',nunique_tract)

In [ ]:
# Tracts present in each dataset
lihtc_tracts = set(lihtc["tract_id"].unique())
lodes_tracts = set(lodes["tract_id"].unique())
acs_tracts = set(controls['geoid'].unique())

# LIHTC tracts missing in LODES
missing_in_lodes = lihtc_tracts - lodes_tracts
missing_in_acs = lihtc_tracts - acs_tracts

lodes_missing_in_acs = lodes_tracts - acs_tracts
acs_missing_in_lodes = acs_tracts - lodes_tracts

print(f"{len(missing_in_lodes):,} LIHTC tracts not found in LODES.")
if missing_in_lodes:
    print("Example missing tract_ids:", list(sorted(missing_in_lodes))[:20])

print(f"{len(missing_in_acs):,} LIHTC tracts not found in ACS.")
if missing_in_acs:
    print("Example missing tract_ids:", list(sorted(missing_in_acs))[:20])

print(f"{len(lodes_missing_in_acs):,} LODES tracts not found in ACS.")
if lodes_missing_in_acs:
    print("Example missing tract_ids:", list(sorted(lodes_missing_in_acs))[:20])

print(f"{len(acs_missing_in_lodes):,} ACS tracts not found in LODES.")
if acs_missing_in_lodes:
    print("Example missing tract_ids:", list(sorted(acs_missing_in_lodes))[:20])
    
print(f'unique tracts in LIHTC:', len(lihtc_tracts))
print(f'unique tracts in LODES:', len(lodes_tracts))
print(f'unique tracts in ACS:', len(acs_tracts))


In [ ]:
lihtc['ever_treated'].value_counts()
# lihtc['treated'].value_counts()
